In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:
%load_ext lab_black

In [3]:
path_counts_reference = (
    "../../data/HECA/reference/endometriumAtlasV2_cells_with_counts.h5ad"
)
path_counts_query = "../../data/HECA/queries_counts/Asherman_syndrome_wo_wang.h5ad"

In [4]:
path_output_merged_adata = (
    "../../data/HECA/queries_counts/HECA_ref_plus_Santamaria_query.h5ad"
)

### Load and clean up adata_ref:

In [5]:
adata_ref = sc.read_h5ad(path_counts_reference)

remove disease data from reference:

In [6]:
adata_ref = adata_ref[adata_ref.obs.Endometrial_pathology == "C", :].copy()

remove dataset-specific info from var:

In [7]:
var_cols_to_remove = list()
for col in adata_ref.var.columns:
    if "-" in col:
        var_cols_to_remove.append(col)
adata_ref.var.drop(columns=var_cols_to_remove, inplace=True)

In [8]:
del adata_ref.uns
del adata_ref.obsp

In [9]:
adata_ref.obs["reference_or_query"] = "reference"

set .X to raw counts instead of normalized

In [10]:
adata_ref.X = adata_ref.layers["counts"]

In [11]:
del adata_ref.layers["counts"]
del adata_ref.obsm

In [12]:
adata_ref.shape

(168434, 17736)

In [13]:
adata_ref

AnnData object with n_obs × n_vars = 168434 × 17736
    obs: 'n_genes', 'sample', 'library', 'Processing', 'Treatment', '10x kit', 'percent_mito', 'n_counts', 'scrublet_score', 'genotype', 'Library_genotype', 'Group', 'Endometriosis_stage', 'Hormonal treatment', 'Binary Stage', 'Stage', 'phase', 'dataset', 'Biopsy_type', 'Tissue_sampled', 'Age', 'Endometrial_pathology', 'celltype', 'lineage', 'label_long', 'reference_or_query'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'

In [14]:
adata_ref.obs["dataset"].unique()

['Mareckova', 'GarciaAlonso', 'Wang', 'Tan', 'Huang', 'Lai']
Categories (6, object): ['Mareckova', 'GarciaAlonso', 'Wang', 'Tan', 'Huang', 'Lai']

## Load and clean adata_query:

In [15]:
adata_query = sc.read_h5ad(path_counts_query)

rename obs columns to correspond with ref obs columns where possible and needed:

In [16]:
obs_col_renaming = {
    "nCount_RNA": "n_counts",
    "mitochondria_ratio": "percent_mito",
    "celltype": "celltype_query",
    "principal_cell_types": "principal_celltype_query",
}

In [17]:
adata_query.obs.rename(columns=obs_col_renaming, inplace=True)

remove unexpressed genes to limit computational time when merging ref and query:

In [18]:
adata_query.shape

(59301, 33538)

In [19]:
sc.pp.filter_genes(adata_query, min_cells=1)

In [20]:
adata_query.shape

(59301, 29963)

add ref/query info:

In [21]:
adata_query.obs["reference_or_query"] = "query"

## Merge reference and query:

In [22]:
adata = sc.concat(adatas=(adata_ref, adata_query), join="outer", index_unique=None)

In [23]:
adata

AnnData object with n_obs × n_vars = 227735 × 29989
    obs: 'n_genes', 'sample', 'library', 'Processing', 'Treatment', '10x kit', 'percent_mito', 'n_counts', 'scrublet_score', 'genotype', 'Library_genotype', 'Group', 'Endometriosis_stage', 'Hormonal treatment', 'Binary Stage', 'Stage', 'phase', 'dataset', 'Biopsy_type', 'Tissue_sampled', 'Age', 'Endometrial_pathology', 'celltype', 'lineage', 'label_long', 'reference_or_query', 'nFeature_RNA', 'menstrual_cycle_day', 'principal_celltype_query', 'celltype_query'

filter genes:

In [24]:
adata.shape

(227735, 29989)

In [25]:
sc.pp.filter_genes(adata, min_cells=5)

In [26]:
adata.shape

(227735, 27131)

To prevent saving errors:

In [27]:
for cat in adata.obs.columns:
    if isinstance(adata.obs[cat].values, pd.Categorical):
        pass
    elif pd.api.types.is_float_dtype(adata.obs[cat]):
        pass
    else:
        print(
            f"Setting obs column {cat} (not categorical neither float) to strings to prevent writing error."
        )
        adata.obs[cat] = adata.obs[cat].astype(str)

Setting obs column reference_or_query (not categorical neither float) to strings to prevent writing error.


### Normalize all data together:

In [28]:
median_total_counts = np.median(adata.obs.n_counts)

In [29]:
median_total_counts

9372.0

store raw counts:

In [30]:
adata.layers["counts"] = adata.X

normalize total:

In [31]:
sc.pp.normalize_total(adata, target_sum=median_total_counts)

log-transform:

In [32]:
sc.pp.log1p(adata)

## Add back HVG info from reference:

In [33]:
adata.var["highly_variable"] = False
adata.var.loc[
    adata_ref.var.loc[adata_ref.var.highly_variable, :].index, "highly_variable"
] = True

In [34]:
# adata.var.highly_variable = adata.var.highly_variable.astype(str)

## Store result:

In [36]:
adata.write(path_output_merged_adata)